# Temporal search experiment visualization

This notebook reproduces the figures used to answer RQ1: *what are the trade-offs of retrieving videos first and refining frames afterwards?*  It reads only the saved outputs in `data/experiments/results/exp1_temporal_search_v2/` and writes publication-ready PDF and PNG figures to `notebooks/experiments/visualization/figures/`.

The comparison is a system-level comparison of the recorded runs.  It is not a controlled ablation of only the temporal strategy because the saved planner outputs can differ across methods.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Locate the repository whether the notebook is launched from the root or this directory.
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / 'data' / 'experiments' / 'results' / 'exp1_temporal_search_v2').exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError('Could not locate the repository root.')

RESULT_DIR = ROOT / 'data' / 'experiments' / 'results' / 'exp1_temporal_search_v2'
FIG_DIR = ROOT / 'notebooks' / 'experiments' / 'visualization' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

METHODS = ['ATS', 'Vortex', 'DEV']
COLORS = {'ATS': '#377eb8', 'Vortex': '#4daf4a', 'DEV': '#e41a1c'}
KS = [1, 5, 10, 20, 50, 100]
plt.rcParams.update({'font.size': 9, 'axes.labelsize': 9, 'axes.titlesize': 10,
                     'legend.fontsize': 8, 'pdf.fonttype': 42, 'ps.fonttype': 42})

In [ ]:
summary = pd.read_csv(RESULT_DIR / 'temporal_search_summary.csv')
summary['method'] = pd.Categorical(summary['method'], METHODS, ordered=True)
summary = summary.sort_values('method').reset_index(drop=True)

# Per-query files permit an analysis limited to temporal KIS queries.
answer_files = {
    'ATS': RESULT_DIR / 'ats' / 'ats_answers_top100.csv',
    'Vortex': RESULT_DIR / 'vortex' / 'vortex_answers_top100.csv',
    'DEV': RESULT_DIR / 'dev' / 'dev_answers_top100.csv',
}
answers = pd.concat(
    [pd.read_csv(path).assign(method=method) for method, path in answer_files.items()],
    ignore_index=True,
)
answers['rank'] = pd.to_numeric(answers['rank'], errors='coerce')
answers['query_type'] = answers['query_type'].str.upper()

with open(RESULT_DIR / 'experiment_manifest.json', encoding='utf-8') as f:
    manifest = json.load(f)

display(summary)
print(f"Queries: {len(answers.query_id.unique())}; KIS: {(answers.query_type == 'KIS').sum() // len(METHODS)}")
print(f"Figures will be written to: {FIG_DIR}")

In [ ]:
def save_figure(fig, stem):
    """Save both a vector PDF for LaTeX and a PNG for quick inspection."""
    fig.savefig(FIG_DIR / f'{stem}.pdf', bbox_inches='tight')
    fig.savefig(FIG_DIR / f'{stem}.png', dpi=300, bbox_inches='tight')

def recall_at(frame, k, query_type=None):
    subset = frame if query_type is None else frame[frame['query_type'] == query_type]
    return (subset['rank'].le(k).mean() * 100)

def strategy_label(method):
    return summary.loc[summary['method'].eq(method), 'temporal_strategy'].iloc[0]

## Figure 1 — end-to-end effectiveness versus latency

The left panel shows all 78 evaluated queries, matching the aggregate metric reported by the experiment.  The right panel uses median end-to-end latency; it therefore includes planning and retrieval time.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.1, 2.65), gridspec_kw={'width_ratios': [1.35, 1]})

for _, row in summary.iterrows():
    axes[0].plot(KS, [row[f'R@{k}'] for k in KS], marker='o', ms=4, lw=1.8,
                 color=COLORS[row['method']], label=row['method'])
axes[0].set_xscale('symlog', linthresh=5)
axes[0].set_xticks(KS)
axes[0].set_xticklabels([str(k) for k in KS])
axes[0].set_ylim(25, 100)
axes[0].set_xlabel('Retrieval depth $k$')
axes[0].set_ylabel('Recall@k (%)')
axes[0].set_title('(a) Recall-depth curve')
axes[0].grid(axis='y', alpha=.25)
axes[0].legend(frameon=False, ncol=3, loc='lower right')

latency_s = summary['median_latency_ms'] / 1000
bars = axes[1].bar(summary['method'], latency_s, color=[COLORS[m] for m in summary['method']], width=.62)
axes[1].bar_label(bars, labels=[f'{v:.1f}s' for v in latency_s], padding=2, fontsize=8)
axes[1].set_ylim(0, latency_s.max() * 1.2)
axes[1].set_ylabel('Median end-to-end latency (s)')
axes[1].set_title('(b) Cost of the strategy')
axes[1].grid(axis='y', alpha=.25)

fig.tight_layout(w_pad=2.2)
save_figure(fig, 'temporal_tradeoff')
plt.show()

## Figure 2 — temporal-query head accuracy

KIS is the temporal-query subset.  This figure isolates the operating region most relevant to temporal ordering, at the two shallow depths asked about in the research question.

In [ ]:
kis = answers[answers['query_type'].eq('KIS')].copy()
kis_rows = []
for method in METHODS:
    method_rows = kis[kis['method'].eq(method)]
    for k in (1, 5):
        kis_rows.append({'method': method, 'k': k, 'recall': recall_at(method_rows, k), 'n': len(method_rows)})
kis_recall = pd.DataFrame(kis_rows)

fig, axes = plt.subplots(1, 2, figsize=(6.4, 2.65), sharey=True)
for ax, k in zip(axes, (1, 5)):
    panel = kis_recall[kis_recall['k'].eq(k)].set_index('method').loc[METHODS].reset_index()
    bars = ax.bar(panel['method'], panel['recall'], color=[COLORS[m] for m in panel['method']], width=.62)
    ax.bar_label(bars, labels=[f'{v:.1f}' for v in panel['recall']], padding=2, fontsize=8)
    ax.set_title(f'KIS Recall@{k} (n={panel["n"].iloc[0]})')
    ax.set_xlabel('Method')
    ax.set_ylim(0, 80)
    ax.grid(axis='y', alpha=.25)
axes[0].set_ylabel('Recall (%)')
fig.tight_layout(w_pad=1.1)
save_figure(fig, 'temporal_kis_head')
plt.show()
display(kis_recall.pivot(index='method', columns='k', values='recall').rename(columns={1: 'KIS R@1', 5: 'KIS R@5'}))

## Figure 3 — where DEV gains and loses rank

Positive values mean DEV has higher recall than the comparator.  The rank-band panel exposes whether misses at shallow depth reappear later in the list.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.1, 2.65), gridspec_kw={'width_ratios': [1.25, 1]})
dev = summary.set_index('method').loc['DEV']
for comparator, marker in [('ATS', 'o'), ('Vortex', 's')]:
    base = summary.set_index('method').loc[comparator]
    delta = [dev[f'R@{k}'] - base[f'R@{k}'] for k in KS]
    axes[0].plot(KS, delta, marker=marker, lw=1.8, ms=4, color=COLORS[comparator], label=f'DEV - {comparator}')
axes[0].axhline(0, color='black', lw=.8)
axes[0].set_xscale('symlog', linthresh=5)
axes[0].set_xticks(KS)
axes[0].set_xticklabels([str(k) for k in KS])
axes[0].set_xlabel('Retrieval depth $k$')
axes[0].set_ylabel('Recall difference (percentage points)')
axes[0].set_title('(a) DEV relative recall')
axes[0].grid(axis='y', alpha=.25)
axes[0].legend(frameon=False)

band_labels = ['1--5', '6--10', '11--20', '21--100', 'not found']
def rank_band(rank):
    if pd.isna(rank) or rank > 100: return 'not found'
    if rank <= 5: return '1--5'
    if rank <= 10: return '6--10'
    if rank <= 20: return '11--20'
    return '21--100'
band_counts = (answers.assign(rank_band=answers['rank'].map(rank_band))
               .groupby(['method', 'rank_band']).size().unstack(fill_value=0)
               .reindex(index=METHODS, columns=band_labels, fill_value=0))
bottom = np.zeros(len(METHODS))
band_colors = ['#2166ac', '#67a9cf', '#fddbc7', '#ef8a62', '#bdbdbd']
for label, color in zip(band_labels, band_colors):
    axes[1].bar(METHODS, band_counts[label], bottom=bottom, label=label, color=color, width=.68)
    bottom += band_counts[label].to_numpy()
axes[1].set_ylabel('Number of queries')
axes[1].set_title('(b) First relevant-result rank')
axes[1].legend(title='Rank band', frameon=False, fontsize=7, title_fontsize=8, loc='upper left')
axes[1].grid(axis='y', alpha=.25)

fig.tight_layout(w_pad=2.1)
save_figure(fig, 'temporal_dev_rank_tradeoff')
plt.show()
display(band_counts)

## Copy-ready table and report numbers

The next cell prints a compact `booktabs` table and the exact values used in the report.  The generated figures are sufficient for the LaTeX snippet: Figure 1 is the headline recall--latency trade-off and Figure 2 is optional supporting evidence for the temporal (KIS) subset.

In [ ]:
paper_table = summary[['method', 'R@1', 'R@5', 'R@10', 'R@100', 'median_latency_ms']].copy()
paper_table['median_latency_ms'] = paper_table['median_latency_ms'] / 1000
paper_table = paper_table.rename(columns={'method': 'Method', 'median_latency_ms': 'Median latency (s)'})
display(paper_table)
print(paper_table.to_latex(index=False, float_format=lambda x: f'{x:.2f}', escape=False))

ats_latency = summary.set_index('method').loc['ATS', 'median_latency_ms']
vortex_latency = summary.set_index('method').loc['Vortex', 'median_latency_ms']
dev_latency = summary.set_index('method').loc['DEV', 'median_latency_ms']
print(f'DEV latency multiplier: {dev_latency / ats_latency:.2f}x ATS; {dev_latency / vortex_latency:.2f}x Vortex.')
print('Manifest configuration:', manifest.get('config', manifest))